# Semana 9 · Limpieza del dataset de Spotify con pandas

**Caso:** Spotify — capacidad de streaming y curaduría regional · **Autor:** Julián Andrés Solano Ledesma

El CSV de Kaggle no trae nulos ni duplicados, así que primero se **ensucia de forma reproducible** (`ensuciar_spotify.py`, semilla 2026). Después se limpia con `limpieza_spotify.py`, que aplica reglas generales sin saber qué se inyectó. El cuaderno documenta:

1. el **antes**: nulos, duplicados, tipos y formatos;
2. los **7 pasos** de limpieza con su efecto;
3. el **después** y la tabla comparativa;
4. una **verificación contra el CSV original**: la limpieza no alteró ningún dato que no debía.

## 1. Generar la versión sucia (reproducible)

In [1]:
from pathlib import Path

import pandas as pd

import ensuciar_spotify as E
import limpieza_spotify as L

original = pd.read_csv(E.ORIGEN)
sucio, inyectado = E.ensuciar(original)
Path("datos").mkdir(exist_ok=True)
sucio.to_csv(E.DESTINO, index=False)
print(f"{len(original)} filas originales -> {len(sucio)} filas sucias")
inyectado

500 filas originales -> 520 filas sucias


,problema,columna,celdas o filas,cómo se inyectó
0,Formato de texto,Country,50,mayúsculas/minúsculas y espacios sobrantes
1,Formato de texto,Genre,30,mayúsculas/minúsculas y espacios sobrantes
2,Formato de texto,Platform Type,30,mayúsculas/minúsculas y espacios sobrantes
3,Formato de texto,Artist,15,mayúsculas/minúsculas y espacios sobrantes
4,Sinónimos,Country,7,"""USA"" y ""UK"" en lugar del nombre completo"
5,Sinónimos,Genre,10,"""Hip-Hop"" y ""KPOP"""
6,Tipo incorrecto,Total Streams (Millions),40,"separador de miles: ""3,695.53"""
7,Tipo incorrecto,Skip Rate (%),60,"símbolo de porcentaje: ""2.24%"""
8,Faltante,Monthly Listeners (Millions),25,vacío
9,Faltante,Skip Rate (%),15,vacío


## 2. Antes: diagnóstico del archivo sucio

Se lee como lo haría cualquier análisis, reconociendo los marcadores de nulo (`"N/A"`, `"-"`). Revisar nulos, tipos y duplicados antes de analizar es el primer paso.

In [2]:
antes = pd.read_csv(E.DESTINO, na_values=L.MARCAS_NULO)
texto_orig = ["Country", "Artist", "Album", "Genre", "Platform Type"]

print("Nulos por columna:")
print(antes.isnull().sum()[lambda s: s > 0].to_string())
print(f"\nFilas duplicadas exactas: {antes.duplicated().sum()}")

Nulos por columna:
Country                             6
Genre                               4
Monthly Listeners (Millions)       25
Avg Stream Duration (Min)           6
Streams Last 30 Days (Millions)    11
Skip Rate (%)                      15

Filas duplicadas exactas: 12


In [3]:
# Tipos: dos columnas numéricas llegaron como texto
antes.dtypes.to_frame("tipo").T

,Country,Artist,Album,Genre,Release Year,Monthly Listeners (Millions),Total Streams (Millions),Total Hours Streamed (Millions),Avg Stream Duration (Min),Platform Type,Streams Last 30 Days (Millions),Skip Rate (%)
tipo,str,str,str,str,int64,float64,str,float64,float64,str,float64,str


In [4]:
# Formatos: el mismo plan escrito de 10 formas y el país de muchas más
print(antes["Platform Type"].value_counts().rename(index=repr).to_string())   # repr muestra los espacios
print("\nEjemplos de variantes del país:",
      sorted(v for v in antes["Country"].dropna().unique() if v.strip() != v or v.isupper() or v.islower())[:8])

Platform Type
'Free'          249
'Premium'       242
'FREE'            7
'premium'         5
'Premium   '      5
'free'            4
'  Premium '      3
'PREMIUM'         2
'  Free '         2
'Free   '         1

Ejemplos de variantes del país: ['  Argentina ', '  France ', '  India ', '  Indonesia ', '  Italy ', '  Mexico ', '  Netherlands ', '  Russia ']


## 3. Limpieza: 7 pasos documentados

In [5]:
limpio, bitacora = L.limpiar(L.leer_sucio(E.DESTINO))
limpio.to_csv(L.SALIDA, index=False)
bitacora

,paso,dimensión,detalle,filas antes,filas después
0,1. Nombres de columna,Consistencia,12 columnas a snake_case (modelo Semana 6),520,520
1,2. Formato de texto,Consistencia,139 celdas llevadas a su valor oficial; 0 fuer...,520,520
2,3. Tipos,Exactitud,6 medidas a float64 y el año a Int16 (0 celdas...,520,520
3,4. Duplicados,Unicidad,20 filas repetidas eliminadas,520,500
4,5. Faltantes,Completitud,10 filas sin país/género eliminadas; 54 medida...,500,490
5,6. Año por álbum,Consistencia,361 celdas corregidas a un año por álbum,490,490
6,7. Valores imposibles,Exactitud,2 filas con streams 30 d > streams totales,490,488


## 4. Después: el archivo limpio

In [6]:
print("Nulos por columna:", int(limpio.isnull().sum().sum()))
print("Filas duplicadas exactas:", int(limpio.duplicated().sum()))
print("\nPlan:", limpio["plan"].value_counts().to_dict())
print("Países distintos:", limpio["pais"].nunique(), "· Géneros:", limpio["genero"].nunique())
limpio.dtypes.to_frame("tipo").T

Nulos por columna: 0
Filas duplicadas exactas: 0

Plan: {'Premium': 246, 'Free': 242}
Países distintos: 20 · Géneros: 10


,pais,artista,album,genero,anio_lanzamiento,oyentes_mensuales_m,streams_totales_m,horas_totales_m,duracion_prom_min,plan,streams_30d_m,skip_rate_pct
tipo,category,category,category,category,Int16,float64,float64,float64,float64,category,float64,float64


## 5. Comparación antes / después

In [7]:
def extra(df, album, anio, s30, total):
    return {
        "Álbumes con más de un año de lanzamiento": int((df.groupby(album)[anio].nunique() > 1).sum()),
        "Filas imposibles (streams 30 d > totales)": int((pd.to_numeric(df[s30], errors="coerce")
                                                         > L.a_numero(df[total].astype(str))).sum()),
    }

p_antes = {**L.perfil(antes, texto_orig),
           **extra(antes, "Album", "Release Year", "Streams Last 30 Days (Millions)", "Total Streams (Millions)")}
p_despues = {**L.perfil(limpio, L.TEXTO),
             **extra(limpio, "album", "anio_lanzamiento", "streams_30d_m", "streams_totales_m")}
p_despues["Memoria (KB)"] = round(limpio.memory_usage(deep=True).sum() / 1024, 1)
p_antes["Memoria (KB)"] = round(antes.memory_usage(deep=True).sum() / 1024, 1)

comparacion = pd.DataFrame({"Antes": pd.Series(p_antes, dtype=object),
                            "Después": pd.Series(p_despues, dtype=object)})
comparacion

,Antes,Después
Filas,520,488
Celdas nulas,67,0
Filas duplicadas exactas,12,0
Columnas numéricas (de 7),5,7
Celdas de texto con espacios sobrantes,53,0
Formas distintas de escribir el país,63,20
Formas distintas de escribir el plan,10,2
Álbumes con más de un año de lanzamiento,15,0
Filas imposibles (streams 30 d > totales),2,0
Memoria (KB),224.5,30.4


## 6. Verificación contra el CSV original

Como el archivo sucio se generó a partir del original, se puede comprobar que la limpieza **no alteró datos válidos**. Cada fila limpia se empareja con su fila original por (`album`, `horas_totales_m`), una combinación única en el original. Las únicas diferencias admitidas son las celdas imputadas y los años unificados por álbum.

In [8]:
ref = original.rename(columns=L.COLUMNAS)
assert not ref.duplicated(["album", "horas_totales_m"]).any()

par = limpio.astype({c: "string" for c in L.TEXTO}).merge(
    ref.astype({c: "string" for c in L.TEXTO}),
    on=["album", "horas_totales_m"], suffixes=("", "_orig"), validate="one_to_one")
print(f"Filas limpias emparejadas con el original: {len(par)} de {len(limpio)}")

dif_texto = {c: int((par[c] != par[f"{c}_orig"]).sum()) for c in ["pais", "artista", "genero", "plan"]}
medidas = [c for c in L.MEDIDAS if c != "horas_totales_m"]
dif_medidas = {c: int((par[c] - par[f"{c}_orig"]).abs().gt(0.005).sum()) for c in medidas}
print("Diferencias en texto:", dif_texto)
print("Diferencias en medidas:", dif_medidas, "-> total", sum(dif_medidas.values()))
print("Medidas imputadas según la bitácora:", bitacora.loc[4, "detalle"])

Filas limpias emparejadas con el original: 488 de 488
Diferencias en texto: {'pais': 0, 'artista': 0, 'genero': 0, 'plan': 0}
Diferencias en medidas: {'oyentes_mensuales_m': 24, 'streams_totales_m': 0, 'duracion_prom_min': 5, 'streams_30d_m': 10, 'skip_rate_pct': 15} -> total 54
Medidas imputadas según la bitácora: 10 filas sin país/género eliminadas; 54 medidas imputadas (mediana del álbum)


**Lectura:** el texto coincide al 100 % con el original. Las únicas diferencias numéricas son celdas que llegaron vacías y se imputaron con la mediana del álbum: la limpieza no tocó ningún valor válido.

## 7. Qué filas se eliminaron y por qué

In [9]:
eliminadas = len(sucio) - len(limpio)
print(f"{len(sucio)} filas sucias - {eliminadas} eliminadas = {len(limpio)} filas limpias")
print(bitacora[["paso", "filas antes", "filas después"]]
      .assign(eliminadas=lambda b: b["filas antes"] - b["filas después"])
      .query("eliminadas > 0").to_string(index=False))

520 filas sucias - 32 eliminadas = 488 filas limpias
                 paso  filas antes  filas después  eliminadas
        4. Duplicados          520            500          20
         5. Faltantes          500            490          10
7. Valores imposibles          490            488           2
